# 01 - Data Exploration

Explore NASA SSC spacecraft position data and OMNI solar wind data.

**Spacecraft:** ISS (LEO), DSCOVR (L1), MMS-1 (magnetosphere)  
**Coordinate Systems:** GSE (Geocentric Solar Ecliptic), GEO (Geographic)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go

from src.data.ssc_client import SSCClient
from src.data.solar_wind import SolarWindClient

import os
os.chdir('..')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Available Spacecraft

In [ ]:
ssc = SSCClient()
observatories = ssc.list_observatories()
print(f'Total observatories: {len(observatories)}')
observatories.head(20)

## 2. Fetch Spacecraft Positions

In [ ]:
# Fetch a short test period first (1 week)
iss_df = ssc.fetch_positions('iss', '2024-01-01', '2024-01-08')
print(f'ISS data points: {len(iss_df)}')
print(f'Columns: {list(iss_df.columns)}')
iss_df.head()

In [ ]:
dscovr_df = ssc.fetch_positions('dscovr', '2024-01-01', '2024-01-08')
print(f'DSCOVR data points: {len(dscovr_df)}')
dscovr_df.head()

In [ ]:
mms_df = ssc.fetch_positions('mms1', '2024-01-01', '2024-01-08')
print(f'MMS-1 data points: {len(mms_df)}')
mms_df.head()

## 3. 3D Orbit Visualization

In [ ]:
from src.utils.visualization import plot_3d_orbit_plotly

# ISS orbit (LEO)
if len(iss_df) > 0 and 'x_gse' in iss_df.columns:
    positions = iss_df[['x_gse', 'y_gse', 'z_gse']].values
    fig = plot_3d_orbit_plotly(positions, title='ISS Orbit (1 week, GSE coordinates)')
    fig.show()

In [ ]:
# Compare all three orbits
fig = go.Figure()

datasets = [
    ('ISS (LEO)', iss_df, 'blue'),
    ('DSCOVR (L1)', dscovr_df, 'green'),
    ('MMS-1 (HEO)', mms_df, 'red'),
]

for name, df, color in datasets:
    if len(df) > 0 and 'x_gse' in df.columns:
        fig.add_trace(go.Scatter3d(
            x=df['x_gse'].values, y=df['y_gse'].values, z=df['z_gse'].values,
            mode='lines', name=name, line=dict(color=color, width=2),
        ))

fig.update_layout(
    title='All Spacecraft Orbits (1 week)',
    scene=dict(xaxis_title='X GSE (km)', yaxis_title='Y GSE (km)', zaxis_title='Z GSE (km)'),
    width=900, height=700,
)
fig.show()

## 4. Data Quality Check

In [ ]:
for name, df in [('ISS', iss_df), ('DSCOVR', dscovr_df), ('MMS-1', mms_df)]:
    if len(df) > 0:
        print(f'\n{name}:')
        print(f'  Time range: {df["time"].min()} to {df["time"].max()}')
        time_diffs = df['time'].diff().dt.total_seconds().dropna()
        print(f'  Time resolution: median={time_diffs.median():.0f}s, max gap={time_diffs.max():.0f}s')
        print(f'  Missing values: {df.isnull().sum().to_dict()}')
        print(f'  Stats:')
        numeric_cols = [c for c in df.columns if c.startswith(('x_', 'y_', 'z_'))]
        if numeric_cols:
            print(df[numeric_cols].describe().round(2))

## 5. Solar Wind Data

In [ ]:
solar_client = SolarWindClient()
solar_df = solar_client.fetch_solar_wind('2024-01-01', '2024-01-08')
print(f'Solar wind data points: {len(solar_df)}')
print(f'Columns: {list(solar_df.columns)}')
solar_df.head()

In [ ]:
# Plot solar wind parameters
if len(solar_df) > 0:
    param_cols = [c for c in solar_df.columns if c != 'time']
    n_params = min(len(param_cols), 6)
    
    fig, axes = plt.subplots(n_params, 1, figsize=(14, 3 * n_params), sharex=True)
    if n_params == 1:
        axes = [axes]
    
    for ax, col in zip(axes, param_cols[:n_params]):
        ax.plot(solar_df['time'], solar_df[col], linewidth=0.5)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Time')
    plt.suptitle('Solar Wind Parameters (1 week)', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Geocentric Distance Over Time

In [ ]:
from src.utils.coords import geocentric_distance

fig, ax = plt.subplots(figsize=(14, 5))

for name, df, color in datasets:
    if len(df) > 0 and 'x_gse' in df.columns:
        r = geocentric_distance(df['x_gse'].values, df['y_gse'].values, df['z_gse'].values)
        ax.plot(df['time'], r, label=f'{name} (mean={r.mean():.0f} km)', color=color, linewidth=0.8)

ax.axhline(y=6371, color='gray', linestyle='--', alpha=0.5, label='Earth surface')
ax.set_xlabel('Time')
ax.set_ylabel('Distance from Earth center (km)')
ax.set_title('Geocentric Distance')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()